# Does confidence fall when decisive information is missing?
This notebook evaluates Qwen3-4B-Instruct-2507 on matched complete, underspecified, and misleading policy questions.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes pandas

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload ambiguity_eval_implicit.csv
import io, pandas as pd
csv_name = next(name for name in uploaded if name.lower().endswith('.csv'))
df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
required = {'id','base_id','category','variant','policy','case','question','expected_decision'}
assert required.issubset(df.columns), required - set(df.columns)
assert len(df) == 60
print(df.variant.value_counts())
print(df.expected_decision.value_counts())
df.head()

In [ ]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_config, device_map='auto'
).eval()
MODEL_REVISION = getattr(model.config, '_commit_hash', None)
print('transformers', transformers.__version__)
print('model revision', MODEL_REVISION)
print('device', model.device)

In [ ]:
import json, re

SYSTEM = '''You are helping decide whether a case satisfies a written policy. Return one JSON object and nothing else. The schema is: {"decision": "YES|NO|INSUFFICIENT", "confidence": 0, "missing_information": "short text or NONE"}. Choose the decision using your best judgment. Confidence must be an integer from 0 to 100. If your decision is INSUFFICIENT, briefly state what information you need; otherwise use NONE.'''

def parse_json(text):
    text = text.strip()
    match = re.search(r'\{.*?\}', text, flags=re.S)
    if not match:
        return {'decision':'PARSE_ERROR','confidence':None,'missing_information':'','parse_error':True}
    try:
        obj = json.loads(match.group(0))
        decision = str(obj.get('decision','')).upper().strip()
        confidence = int(float(obj.get('confidence')))
        if decision not in {'YES','NO','INSUFFICIENT'} or not 0 <= confidence <= 100:
            raise ValueError('invalid values')
        return {'decision':decision,'confidence':confidence,'missing_information':str(obj.get('missing_information','')),'parse_error':False}
    except Exception:
        return {'decision':'PARSE_ERROR','confidence':None,'missing_information':'','parse_error':True}

def evaluate_row(row):
    user = f'''Policy: {row.policy}

Case: {row['case']}

Question: {row.question}'''
    messages = [{'role':'system','content':SYSTEM},{'role':'user','content':user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    raw = tokenizer.decode(generated[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    parsed = parse_json(raw)
    return raw, parsed


In [ ]:
rows = []
for i, row in df.iterrows():
    raw, parsed = evaluate_row(row)
    rows.append({
        **row.to_dict(),
        'prediction': parsed['decision'],
        'confidence': parsed['confidence'],
        'model_missing_information': parsed['missing_information'],
        'parse_error': parsed['parse_error'],
        'raw_output': raw,
        'correct': parsed['decision'] == row.expected_decision,
    })
    if (i + 1) % 5 == 0: print(i + 1, '/', len(df))
results = pd.DataFrame(rows)
results.head()

In [ ]:
import numpy as np

results['high_confidence_error'] = (~results.correct) & (results.confidence >= 80)
results['unsupported_commitment'] = results.variant.isin(['underspecified','misleading']) & results.prediction.isin(['YES','NO'])
by_variant = results.groupby('variant').agg(
    n=('id','size'),
    accuracy=('correct','mean'),
    mean_confidence=('confidence','mean'),
    high_confidence_error_rate=('high_confidence_error','mean'),
    unsupported_commitment_rate=('unsupported_commitment','mean'),
).reset_index()

decision_table = pd.crosstab(results.variant, results.prediction)
paired_confidence = results.pivot(index='base_id', columns='variant', values='confidence')
paired_confidence['complete_minus_underspecified'] = paired_confidence.complete - paired_confidence.underspecified
paired_confidence['complete_minus_misleading'] = paired_confidence.complete - paired_confidence.misleading

summary = {
    'model': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'n_examples': int(len(results)),
    'overall_accuracy': float(results.correct.mean()),
    'parse_errors': int(results.parse_error.sum()),
    'metrics_by_variant': by_variant.to_dict(orient='records'),
    'decision_counts_by_variant': {k:{str(k2):int(v2) for k2,v2 in v.items()} for k,v in decision_table.to_dict(orient='index').items()},
    'mean_confidence_drop_complete_to_underspecified': float(paired_confidence.complete_minus_underspecified.mean()),
    'mean_confidence_drop_complete_to_misleading': float(paired_confidence.complete_minus_misleading.mean()),
}
print(json.dumps(summary, indent=2))
display(by_variant)
display(decision_table)

results.to_csv('results.csv', index=False)
by_variant.to_csv('metrics_by_variant.csv', index=False)
paired_confidence.to_csv('paired_confidence.csv')
results.loc[~results.correct].to_csv('errors_for_review.csv', index=False)
with open('summary.json','w') as f: json.dump(summary, f, indent=2)

In [ ]:
from google.colab import files
for path in ['results.csv','summary.json','metrics_by_variant.csv','paired_confidence.csv','errors_for_review.csv']:
    files.download(path)